# Revision de conceptos teoricos

In [1]:
import pandas as pd
import os


# Carga de ambos embeddings y del panel de etiquetas
emb_dev     = pd.read_parquet("D:/financial_risk_data/embeddings/emb_dev/embeddings_desarrollo.parquet")
emb_rel     = pd.read_parquet("D:/financial_risk_data/embeddings/emb_rel_dev/erel_desarrollo.parquet")
panel_label = pd.read_parquet("D:/financial_risk_data/processed/panel_tabular_labeled.parquet")

# Normalizar tipos de clave en las tres fuentes
for df in [emb_dev, emb_rel, panel_label]:
    df['CERT']   = df['CERT'].astype(str)
    df['period'] = df['period'].astype(str)

print("=== ESTRUCTURA ===")
print(f"emb_dev shape     : {emb_dev.shape}")
print(f"emb_rel shape     : {emb_rel.shape}")
print(f"panel_label shape : {panel_label.shape}")
print(f"Columnas label    : {panel_label.columns.tolist()}")
print()

# Construcción de claves compuestas
keys_dev   = set(zip(emb_dev['CERT'],   emb_dev['period']))
keys_rel   = set(zip(emb_rel['CERT'],   emb_rel['period']))
keys_label = set(zip(panel_label['CERT'], panel_label['period']))

solo_dev = keys_dev - keys_rel
solo_rel = keys_rel - keys_dev
comun    = keys_dev & keys_rel

print("=== ALINEACIÓN TABULAR vs RELACIONAL ===")
print(f"Claves en tabular             : {len(keys_dev)}")
print(f"Claves en relacional          : {len(keys_rel)}")
print(f"Claves comunes (inner join)   : {len(comun)}")
print(f"Solo en tabular (sin e_rel)   : {len(solo_dev)}")
print(f"Solo en relacional (sin e_dev): {len(solo_rel)}")
print()

# Positivos en claves exclusivas de tabular (desde panel_label)
if len(solo_dev) > 0:
    panel_label['_key'] = list(zip(panel_label['CERT'], panel_label['period']))
    pos_perdidos = panel_label[panel_label['_key'].isin(solo_dev)]['failed'].sum()
    panel_label.drop(columns=['_key'], inplace=True)
    print(f"Positivos que se perderían con inner join: {pos_perdidos}")
else:
    print("Sin claves exclusivas en tabular — inner join no pierde ninguna observación")

print()

# Positivos totales según panel_label dentro del rango de desarrollo
keys_comun_series = pd.DataFrame(list(comun), columns=['CERT', 'period'])
panel_dev = panel_label.merge(keys_comun_series, on=['CERT', 'period'], how='inner')
print("=== POSITIVOS EN CONJUNTO DE DESARROLLO ===")
print(f"Total observaciones en inner join : {len(panel_dev)}")
print(f"Positivos (failed=1)              : {panel_dev['failed'].sum()}")
print(f"Tasa de positivos                 : {panel_dev['failed'].mean():.6f}")
print()

print("=== PERIODOS ===")
print(f"Periodos tabular    : {sorted(emb_dev['period'].unique())}")
print(f"Periodos relacional : {sorted(emb_rel['period'].unique())}")

=== ESTRUCTURA ===
emb_dev shape     : (125575, 194)
emb_rel shape     : (125575, 67)
panel_label shape : (206129, 3)
Columnas label    : ['CERT', 'period', 'failed']

=== ALINEACIÓN TABULAR vs RELACIONAL ===
Claves en tabular             : 125575
Claves en relacional          : 125575
Claves comunes (inner join)   : 125575
Solo en tabular (sin e_rel)   : 0
Solo en relacional (sin e_dev): 0

Sin claves exclusivas en tabular — inner join no pierde ninguna observación

=== POSITIVOS EN CONJUNTO DE DESARROLLO ===
Total observaciones en inner join : 125575
Positivos (failed=1)              : 63
Tasa de positivos                 : 0.000502

=== PERIODOS ===
Periodos tabular    : ['2016Q2', '2016Q3', '2016Q4', '2017Q1', '2017Q2', '2017Q3', '2017Q4', '2018Q1', '2018Q2', '2018Q3', '2018Q4', '2019Q1', '2019Q2', '2019Q3', '2019Q4', '2020Q1', '2020Q2', '2020Q3', '2020Q4', '2021Q1', '2021Q2', '2021Q3', '2021Q4']
Periodos relacional : ['2016Q2', '2016Q3', '2016Q4', '2017Q1', '2017Q2', '2017Q3', '20

In [2]:
# ============================================================================
# BLOQUE 1 — CARGA Y CONCATENACION DE EMBEDDINGS
# ============================================================================

import pandas as pd
import numpy as np

import os
import json
from pathlib import Path
import torch

# 1. Parámetros de reproducibilidad (semillas)

SEED = 42
TORCH_SEED = 42
NP_SEED = 42

# 2. Rutas del sistema de archivos

# Carpeta raíz de datos. Ajusta según tu estructura local.
DATA_ROOT = Path("D:/financial_risk_data")
EMBEDDINGS_ROOT = DATA_ROOT / "embeddings"
EMB_DEV_DIR = EMBEDDINGS_ROOT / "emb_dev"
EMB_REL_DIR = EMBEDDINGS_ROOT / "emb_rel_dev"

# Archivos de entrada
EMB_DEV_PATH = EMB_DEV_DIR / "embeddings_desarrollo.parquet"
EMB_REL_PATH = EMB_REL_DIR / "erel_desarrollo.parquet"
PANEL_LABEL_PATH = DATA_ROOT / "processed" / "panel_tabular_labeled.parquet"

# Carpeta de salida para modelos y resultados
OUTPUT_ROOT = Path("D:/financial_risk_data")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
MODELS_DIR = OUTPUT_ROOT / "models"
RESULTS_DIR = OUTPUT_ROOT / "results"
CHECKPOINTS_DIR = OUTPUT_ROOT / "models_checkpoints"

for d in [MODELS_DIR, RESULTS_DIR, CHECKPOINTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# 3. Parámetros de datos (ventanas temporales, separación train/val)

# Ventana temporal
WINDOW_LENGTH = 4  # T=4 trimestres consecutivos
STRIDE = 1         # Ventana deslizante con paso 1

# Separación train/val
N_VAL_PERIODS = 4  # Últimos 4 trimestres para validación (2021Q1-2021Q4)
# Nota: la frontera exacta se determina en Bloque 2 por period_end

# Clave temporal esperada: 23 periodos de 2016Q2 a 2021Q4
EXPECTED_PERIODS = 23
TOTAL_OBS_ESPERADO = 125575  # Sanity check: sum de todas las observaciones

# 4.  Hiperparámetros de arquitectura (MLPProjection y LSTMEncoder)

# Entrada
D_IN = 256  # d_tab (192) + d_rel (64)


# ----------- 1.1 CARGA -----------
VERBOSE = True
emb_dev = pd.read_parquet(EMB_DEV_PATH)
emb_rel = pd.read_parquet(EMB_REL_PATH)

if VERBOSE:
    print(f"emb_dev shape : {emb_dev.shape}")
    print(f"emb_rel shape : {emb_rel.shape}")

# ----------- 1.2 NORMALIZACIÓN DE TIPOS -----------

for df in [emb_dev, emb_rel]:
    df["CERT"]   = df["CERT"].astype(str)
    df["period"] = df["period"].astype(str)

# ----------- 1.3 IDENTIFICAR COLUMNAS DE EMBEDDING -----------

META_DEV = ["CERT", "period"]
META_REL = ["CERT", "period", "y"]

cols_dev = [c for c in emb_dev.columns if c not in META_DEV]
cols_rel = [c for c in emb_rel.columns if c not in META_REL]

assert len(cols_dev) == 192, f"Se esperaban 192 dims tabulares, hay {len(cols_dev)}"
assert len(cols_rel) == 64,  f"Se esperaban 64 dims relacionales, hay {len(cols_rel)}"

# ----------- 1.4 INNER JOIN -----------
# y viaja desde emb_rel y se renombra a failed para consistencia semántica.
# panel_label no es necesario: verificado que y == failed en 125,575 obs.

emb_concat = pd.merge(
    emb_dev[META_DEV + cols_dev],
    emb_rel[["CERT", "period", "y"] + cols_rel],
    on=["CERT", "period"],
    how="inner",
).rename(columns={"y": "failed"})

# ----------- 1.5 VERIFICACIÓN DE INVARIANTES -----------

n_obs     = len(emb_concat)
n_pos     = int(emb_concat["failed"].sum())
n_nan_emb = emb_concat[cols_dev + cols_rel].isna().sum().sum()
n_nan_lab = emb_concat["failed"].isna().sum()
d_in_real = len(cols_dev) + len(cols_rel)
n_periodos = emb_concat["period"].nunique()

assert n_obs      == 125_575,        f"Observaciones esperadas: 125575, obtenidas: {n_obs}"
assert n_pos      == 63,             f"Positivos esperados: 63, obtenidos: {n_pos}"
assert n_nan_emb  == 0,              f"NaN en columnas de embedding: {n_nan_emb}"
assert n_nan_lab  == 0,              f"NaN en etiqueta failed: {n_nan_lab}"
assert d_in_real  == D_IN,           f"d_in esperado: {D_IN}, obtenido: {d_in_real}"
assert n_periodos == EXPECTED_PERIODS, \
    f"Periodos esperados: {EXPECTED_PERIODS}, obtenidos: {n_periodos}"

if VERBOSE:
    print("\n" + "="*60)
    print("BLOQUE 1 — VERIFICACIÓN DE CONCATENACIÓN DE EMBEDDINGS")
    print("="*60)
    print(f"Observaciones totales : {n_obs:,}")
    print(f"Positivos (failed=1)  : {n_pos}")
    print(f"Tasa de positivos     : {n_pos/n_obs:.6f}")
    print(f"Dimensión de entrada  : {d_in_real}  (= {len(cols_dev)} + {len(cols_rel)})")
    print(f"NaN en embeddings     : {n_nan_emb}")
    print(f"NaN en etiqueta       : {n_nan_lab}")
    print(f"Periodos              : {n_periodos}  ({emb_concat['period'].min()} → {emb_concat['period'].max()})")
    print(f"Bancos únicos         : {emb_concat['CERT'].nunique():,}")
    print("="*60)

# ----------- 1.6 SEPARAR METADATOS Y FEATURES -----------

cols_emb = cols_dev + cols_rel  # 256 columnas, orden: dev primero, rel después

assert len(cols_emb) == D_IN
assert cols_emb[:192] == cols_dev
assert cols_emb[192:] == cols_rel

emb_dev shape : (125575, 194)
emb_rel shape : (125575, 67)

BLOQUE 1 — VERIFICACIÓN DE CONCATENACIÓN DE EMBEDDINGS
Observaciones totales : 125,575
Positivos (failed=1)  : 63
Tasa de positivos     : 0.000502
Dimensión de entrada  : 256  (= 192 + 64)
NaN en embeddings     : 0
NaN en etiqueta       : 0
Periodos              : 23  (2016Q2 → 2021Q4)
Bancos únicos         : 6,176


Tenemos una alineación perfecta en todos los ejes: mismo número de claves, cero observaciones exclusivas en ninguna de las dos fuentes, mismos 23 periodos, y los 63 positivos íntegros preservados en el conjunto de desarrollo.

Esto también confirma algo importante sobre la construcción del pipeline anterior: el `T-GCN` procesó exactamente el mismo universo de observaciones que `TabPFN`, lo cual no era trivial dado que el grafo podría haber excluido bancos con conectividad insuficiente. El hecho de que las 125.575 claves coincidan exactamente indica que el `graph builder` mantuvo todos los bancos del panel de desarrollo como nodos, incluso aquellos con grado bajo.

El vector `X_concat` creado posee las variables identificadoras `(CERT, period, failures)`. Para no introducir _data leakage_ mediante las variables `CERT` y `failures` debemos separar de este vector los metadatos de las variables caracteristicas.  Sea $N=125575$ el número total de muestras tras el inner join:
$$
X_{concat} = [ e_{dev} \parallel e_{rel} ] \in \mathbb{R}^{N \times (192 + 64)}
$$

Donde el operador de concatenación $\parallel$ une los vectores de características puras, resultando en una matriz de entrada estrictamente de dimensión $N \times 256$. Las variables `(CERT, period, failures)` actúan únicamente como índices topológicos en el espacio de bases de datos, no en el espacio vectorial $\mathbb{R}^{256}$.

Hay tres preguntas distintas que conviene separar, porque "redundancia" puede significar cosas distintas y cada una tiene una implicación de diseño diferente.
La primera es redundancia lineal directa: cuánta varianza de e_rel puede explicarse mediante una combinación lineal de e_tab. Esto se mide con CCA (Canonical Correlation Analysis) o, más simple, con una regresión lineal multisalida de e_rel sobre e_tab y midiendo el R² por componente. Si el R² es alto, gran parte de e_rel es recuperable linealmente desde e_tab, y la concatenación aporta poco que un MLP no pueda ya inferir de e_tab solo.
La segunda es redundancia no lineal: si CCA o R² lineal dan bajo pero existe igualmente relación, puede ser que e_rel capture información de e_tab transformada de forma no lineal por el T-GCN (agregación de vecinos, mensajes de grafo), en cuyo caso una medida lineal subestimaría la redundancia real. Para esto se usa más comúnmente una medida como mutual information estimada, o más práctico aquí dado el tamaño de muestra, CKA (Centered Kernel Alignment), que es el estándar en la literatura de comparación de representaciones de redes neuronales y captura similitud más allá de relaciones lineales.
La tercera, más simple y un buen primer filtro antes de las otras dos, es la correlación cruzada por componente entre e_tab y e_rel a nivel agregado, y la matriz de correlación completa entre bloques, que te da una primera intuición visual de si hay bloques de e_tab que predicen directamente bloques de e_rel.

In [3]:
# ============================================================
# DIAGNÓSTICO DE REDUNDANCIA e_tab vs e_rel
# Se ejecuta DESPUÉS del Bloque 1, reutilizando emb_concat y cols_dev/cols_rel
# ya construidos y verificados (n_obs=125575, d_in=256=192+64).
# ============================================================
import numpy as np

# e_tab y e_rel ya están alineados fila a fila porque provienen del mismo
# DataFrame emb_concat (inner join ya verificado en 1.4-1.5)
e_tab = emb_concat[cols_dev].values.astype(np.float64)  # (125575, 192)
e_rel = emb_concat[cols_rel].values.astype(np.float64)  # (125575, 64)

assert e_tab.shape == (125_575, 192)
assert e_rel.shape == (125_575, 64)
assert not np.isnan(e_tab).any()
assert not np.isnan(e_rel).any()

print(f"e_tab shape: {e_tab.shape}  |  e_rel shape: {e_rel.shape}")
print("Alineación garantizada: ambos provienen de emb_concat (mismo inner join)")
print()

# ============================================================
# Ejecutar el diagnóstico completo
# ============================================================
from verificar_redundancia_embeddings import diagnostico_completo_redundancia

diagnostico_completo_redundancia(e_tab, e_rel)

e_tab shape: (125575, 192)  |  e_rel shape: (125575, 64)
Alineación garantizada: ambos provienen de emb_concat (mismo inner join)

DIAGNÓSTICO DE REDUNDANCIA ENTRE e_tab Y e_rel
e_tab shape: (125575, 192)  |  e_rel shape: (125575, 64)

----------------------------------------------------------------------
1. CORRELACIÓN CRUZADA AGREGADA (lineal, componente a componente)
----------------------------------------------------------------------
  shape                                        : (192, 64)
  corr_media_abs                               : 0.1559707123141695
  corr_max_abs                                 : 0.9126227305547577
  corr_mediana_abs                             : 0.10277575958397456
  pct_pares_corr_mayor_0.3                     : 14.908854166666666
  pct_pares_corr_mayor_0.5                     : 4.524739583333334
  max_corr_por_componente_rel_media            : 0.3667914105096619
  max_corr_por_componente_rel_min              : 0.033661619784921444

------------------

Empecemos por la discrepancia más llamativa. CCA da una correlación canónica máxima de 0.99 y media de 0.53 con 7 de 30 componentes por encima de 0.7, lo cual sugiere redundancia sustancial. Pero el R² de la regresión lineal directa da una media de solo 0.37, con la mayoría de componentes de e_rel por debajo de 0.5 de varianza explicada. Esta discrepancia no es un error, es la diferencia metodológica entre ambas técnicas: CCA encuentra las combinaciones lineales óptimas de e_tab que correlacionan al máximo con combinaciones de e_rel, lo cual con 192 dimensiones de entrada y solo 125.575 observaciones tiene margen real de sobreajuste, especialmente en los componentes canónicos de orden alto. El R² con holdout es la medida más conservadora y más fiable de las dos, porque mide si esa relación generaliza fuera de muestra componente a componente, sin margen de ajuste combinatorio. La diferencia entre 0.53 (CCA) y 0.37 (R² holdout) es en sí misma informativa: una parte de lo que CCA "encuentra" es sobreajuste.
El CKA de 0.19 es la pieza más reveladora de las cuatro. CKA es la medida diseñada específicamente para detectar similitud de representaciones más allá de relaciones lineales simples, y un valor de 0.19 es bajo en términos absolutos, muy por debajo del umbral de 0.5 que en la literatura se interpreta como similitud sustancial entre representaciones. Esto es importante porque contradice la hipótesis fuerte que estabas planteando: si e_rel fuera simplemente una transformación de e_dev que preserva la mayor parte de su estructura, CKA debería ser alto, no 0.19.
La lectura conjunta de las cuatro métricas es esta: existe redundancia parcial y localizada, no redundancia global. El R² medio de 0.37 con máximo de 0.87 y mínimo de 0.002 te dice que algunas componentes específicas de e_rel sí son fuertemente predecibles desde e_tab (el T-GCN parece haber preservado cierta información tabular casi sin transformar en algunas dimensiones), mientras que la mayoría de componentes de e_rel contienen información que e_tab no explica linealmente. El CKA bajo confirma que, como estructura global de representación, ambos espacios son sustancialmente distintos.
Esto cambia la recomendación respecto a lo que planteamos antes. Eliminar e_dev completamente y quedarte solo con e_rel descartaría aproximadamente el 63% de la varianza de e_rel que no es explicable desde e_tab, más toda la información de e_dev que no pasó al T-GCN de forma recuperable. La hipótesis de "e_rel domina informacionalmente a e_dev" no está sostenida por estos números. Lo que sí está sostenido es que hay un subconjunto de aproximadamente 6 a 14% de los pares de componentes con redundancia alta (corr>0.5 o R²>0.7), que es precisamente el tipo de redundancia parcial que un MLP con objetivo MSE puede aprovechar para colapsar esas dimensiones específicas sin penalización, mientras debería preservar el resto.
Esto reabre la pregunta de diseño de una forma más precisa: el problema no es que e_tab y e_rel sean enteramente redundantes entre sí, es que existe una fracción de redundancia parcial suficiente para que el optimizador encuentre rentable colapsar esa fracción, y el colapso de varianza observado en el MLP puede estar concentrado precisamente en esas dimensiones redundantes, arrastrando con él (por cómo opera una transformación lineal+no lineal conjunta) parte de la señal no redundante.